# Integrative Analysis of Breast Cancer Susceptibility Loci and Tumor Immune Programs Reveals Immune-Associated Candidate Genes in Triple-Negative Breast Cancer

---
## Notebook 3 - Acquisition and Curation of TCGA-BRCA Transcriptomic and Clinical Data


### Overview

This notebook prepares the transcriptomic and clinical datasets used for downstream integrative analyses.

While the previous notebooks identified candidate breast cancer susceptibility genes from GWAS data, this notebook assembles matched RNA-seq expression profiles and clinical annotations from the TCGA Breast Invasive Carcinoma (TCGA-BRCA) cohort.

The resulting datasets provide the tumor-level molecular context required to investigate how inherited susceptibility genes relate to gene expression and immune biology in triple-negative breast cancer (TNBC).

In [1]:
import os
import requests
import json

import pandas as pd
import numpy as np

### 1. Retrieve TCGA RNA-seq File Manifest

The Genomic Data Commons (GDC) API is queried to identify all primary breast tumor RNA-seq datasets generated using the STAR-Counts workflow.

Only primary tumor samples from the TCGA-BRCA project are requested to ensure consistency across downstream analyses.

The resulting manifest is exported for download using the GDC Data Transfer Tool.

In [2]:
files_endpt = "https://api.gdc.cancer.gov/files"

filters = {
    "op": "and",
    "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": ["TCGA-BRCA"]}},
        {"op": "in", "content": {"field": "files.data_category", "value": ["Transcriptome Profiling"]}},
        {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}},
        {"op": "in", "content": {"field": "files.analysis.workflow_type", "value": ["STAR - Counts"]}},
        {"op": "in", "content": {"field": "cases.samples.sample_type", "value": ["Primary Tumor"]}}
    ]
}

params = {
    "filters": json.dumps(filters),
    "fields": "file_id,file_name,cases.submitter_id,cases.samples.submitter_id",
    "format": "JSON",
    "size": "1200"  
}

response = requests.get(files_endpt, params=params)
data = response.json()

file_list = []
for doc in data["data"]["hits"]:
    file_list.append({
        "file_id": doc["file_id"],
        "file_name": doc["file_name"],
        "patient_id": doc["cases"][0]["submitter_id"],
        "sample_id": doc["cases"][0]["samples"][0]["submitter_id"]
    })

manifest_df = pd.DataFrame(file_list)
manifest_df.to_csv("../data/rna_seq/gdc_manifest.csv", index=False)
manifest_for_client = manifest_df[['file_id']].rename(columns={'file_id': 'id'})
manifest_for_client.to_csv("../data/rna_seq/gdc_manifest.txt", sep="\t", index=False)
print("Manifest text file saved successfully!")

Manifest text file saved successfully!


### 2. Download RNA-seq Count Files

Because RNA-seq count files are too large for direct API retrieval within the notebook, the generated manifest is used with the GDC Data Transfer Tool to download all STAR-count files locally.

After download, the files are assembled into a unified expression matrix.

#### Download TCGA_count files in command line

> Go to the GDC Data Transfer Tool website and download the version for your operating system (Windows, Mac, or Linux).
>
> Move the downloaded gdc-client file into your project repository.
>
> Open your terminal (or command prompt), navigate to your repository folder, and run this single command:
>
>```
>./gdc-client download -m data/gdc_manifest.txt -d data/rna_seq/tcga_counts/
> ```

### 3. Assemble the TCGA Gene Expression Matrix

In [3]:
output_dir = "../data/rna_seq/tcga_counts/"  

sample_frames = []

print("Assembling master raw counts matrix...")

for idx, row in manifest_df.iterrows():
    file_path = os.path.join(output_dir, row["file_id"], row["file_name"])
    
    if not os.path.exists(file_path):
        print(f"Warning: File missing, skipping -> {row['file_name']}")
        continue
        
    df = pd.read_csv(file_path, sep="\t", comment="#", skiprows=[1, 2, 3, 4])
    
    df.columns = [
        "gene_id", "gene_name", "gene_type", 
        "unstranded", "stranded_first", "stranded_second", 
        "tpm_unstranded", "fpkm_unstranded", "fpkm_uq_unstranded"
    ]
    
    counts_series = df.set_index("gene_name")["unstranded"]
    counts_series.name = row["sample_id"]
    sample_frames.append(counts_series)

tcga_raw_counts = pd.concat(sample_frames, axis=1)
print(f"Initial Raw Matrix Shape: {tcga_raw_counts.shape} (Includes pseudogenes, lncRNAs, etc.)")

Assembling master raw counts matrix...
Initial Raw Matrix Shape: (60660, 1111) (Includes pseudogenes, lncRNAs, etc.)


### 4. Restrict Analysis to Protein-Coding Genes
Most downstream differential expression and pathway analyses focus on protein-coding genes because they represent the primary functional products of the genome and are most consistently annotated across biological databases.

Non-protein-coding transcripts are therefore removed, producing a cleaner expression matrix for downstream analyses.

In [4]:
gene_metadata_map = df[["gene_id", "gene_type", "gene_name"]].copy()

protein_coding_genes = gene_metadata_map[gene_metadata_map["gene_type"] == "protein_coding"]
protein_coding_ids = protein_coding_genes["gene_name"].tolist()

tcga_raw_counts = tcga_raw_counts.loc[tcga_raw_counts.index.isin(protein_coding_ids)]

print("Assembly and Filtering Complete!")
print(f"Final Protein-Coding Matrix Shape: {tcga_raw_counts.shape} (Rows/Genes x Columns/Samples)")

tcga_raw_counts.to_csv("../data/rna_seq/tcga_brca_raw_counts.csv")

Assembly and Filtering Complete!
Final Protein-Coding Matrix Shape: (19969, 1111) (Rows/Genes x Columns/Samples)


### 5. Retrieve Clinical and Molecular Metadata

In [5]:
cases_endpt = "https://api.gdc.cancer.gov/cases"

fields_list = [
    "submitter_id",
    "diagnoses.age_at_diagnosis",
    "diagnoses.ajcc_pathologic_stage",
    "follow_ups.molecular_tests.gene_symbol",
    "follow_ups.molecular_tests.test_result",
]

clinical_params = {
    "filters": json.dumps({
        "op": "in",
        "content": {"field": "cases.project.project_id", "value": ["TCGA-BRCA"]}
    }),
    "fields": ",".join(fields_list), 
    "format": "JSON",
    "size": "1200"  
}

print("Fetching combined clinical parameters and molecular assays from GDC...")
response = requests.get(cases_endpt, params=clinical_params, timeout=30)
data = response.json()

parsed_records = []

for hit in data["data"]["hits"]:
    patient_id = hit["submitter_id"]

    diagnoses = hit.get("diagnoses", [])
    stage = "Unknown"
    age = "Unknown"

    if diagnoses:
        age_days = diagnoses[0].get("age_at_diagnosis", None)
        if age_days is not None:
            age = round(age_days / 365.25, 1)

        stage = diagnoses[0].get("ajcc_pathologic_stage", "Unknown")
    
    follow_ups = hit.get("follow_ups", [])
    molecular_data = {"ESR1": "Unknown", "PGR": "Unknown", "ERBB2": "Unknown"}

    for follow_up in follow_ups:
        mol_tests = follow_up.get("molecular_tests", [])
        for test in mol_tests:
            gene = test.get("gene_symbol")
            result = test.get("test_result")
            if gene in molecular_data:
                molecular_data[gene] = result

    parsed_records.append({
        "patient_id": patient_id,
        "age_at_diagnosis": age,
        "pathologic_stage": stage,
        "er_status": molecular_data["ESR1"],
        "pr_status": molecular_data["PGR"],
        "her2_status": molecular_data["ERBB2"]
    })

clinical_df = pd.DataFrame(parsed_records)

print(f"\nExtraction complete! Formatted shape: {clinical_df.shape}")
print(clinical_df.head())

Fetching combined clinical parameters and molecular assays from GDC...

[SUCCESS] Extraction complete! Formatted shape: (1098, 6)
     patient_id age_at_diagnosis pathologic_stage er_status pr_status  \
0  TCGA-E9-A5FL             65.9        Stage IIB  Negative  Negative   
1  TCGA-A2-A1G4             71.1       Stage IIIA  Positive  Positive   
2  TCGA-BH-A0HF             77.3         Stage IA  Positive  Positive   
3  TCGA-AR-A1AS             54.8        Stage IIB  Positive  Positive   
4  TCGA-D8-A13Y             52.1         Stage IA  Positive  Positive   

  her2_status  
0    Negative  
1    Negative  
2    Negative  
3    Negative  
4    Negative  


### 6. Identify Triple-Negative Breast Cancer Samples

Triple-negative breast cancer (TNBC) is clinically defined by the absence of:

- Estrogen receptor (ER)
- Progesterone receptor (PR)
- HER2 receptor

Only patients with definitive receptor-status measurements are retained.

Samples with equivocal, unknown, or incomplete receptor information are excluded to minimize clinical misclassification.

In [6]:
for col in ['er_status', 'pr_status', 'her2_status']:
    clinical_df[col] = clinical_df[col].astype(str).str.strip().str.capitalize()

tnbc_mask = (
    (clinical_df['er_status'] == 'Negative') & 
    (clinical_df['pr_status'] == 'Negative') & 
    (clinical_df['her2_status'] == 'Negative')
)

clinical_df['condition_group'] = np.where(tnbc_mask, 'TNBC', 'Other_Breast_Cancer')

valid_statuses = ['Positive', 'Negative']

invalid_mask = (
    (~clinical_df['er_status'].isin(valid_statuses)) |
    (~clinical_df['pr_status'].isin(valid_statuses)) |
    (~clinical_df['her2_status'].isin(valid_statuses))
)

clinical_df.loc[invalid_mask, 'condition_group'] = 'Excluded_Noise'

print("\n--- Verified Cohort Group Counts (Equivocal Filtered) ---")
print(clinical_df['condition_group'].value_counts())

clinical_df = clinical_df[clinical_df['condition_group'] != 'Excluded_Noise'].copy()

clinical_df = clinical_df[
    (clinical_df["age_at_diagnosis"] != "Unknown")
    & (clinical_df["pathologic_stage"] != "Unknown")
].copy()

clinical_df.to_csv("../data/rna_seq/tcga_clinical_raw.csv", index=False)
print(clinical_df.head(10))


--- Verified Cohort Group Counts (Equivocal Filtered) ---
condition_group
Other_Breast_Cancer    672
Excluded_Noise         290
TNBC                   136
Name: count, dtype: int64
      patient_id age_at_diagnosis pathologic_stage er_status pr_status  \
0   TCGA-E9-A5FL             65.9        Stage IIB  Negative  Negative   
1   TCGA-A2-A1G4             71.1       Stage IIIA  Positive  Positive   
2   TCGA-BH-A0HF             77.3         Stage IA  Positive  Positive   
3   TCGA-AR-A1AS             54.8        Stage IIB  Positive  Positive   
4   TCGA-D8-A13Y             52.1         Stage IA  Positive  Positive   
6   TCGA-LD-A9QF             73.6         Stage IA  Negative  Negative   
7   TCGA-EW-A1IZ             54.0       Stage IIIA  Positive  Positive   
9   TCGA-AO-A1KO             46.0        Stage IIB  Positive  Positive   
10  TCGA-AN-A0XO             59.2       Stage IIIA  Positive  Negative   
11  TCGA-D8-A27K             47.7        Stage IIB  Positive  Positive   

   

### 7. Match RNA-seq Samples to Clinical Records

Transcriptomic and clinical datasets originate from independent sources within TCGA and therefore require careful alignment.

Patient identifiers are matched between both datasets to ensure that every expression profile corresponds to the correct clinical record.

Only patients present in both datasets are retained.

In [7]:
clinical_df = pd.read_csv("../data/rna_seq/tcga_clinical_raw.csv")
tcga_raw_counts = pd.read_csv("../data/rna_seq/tcga_brca_raw_counts.csv")

if "gene_name" in tcga_raw_counts.columns:
    tcga_raw_counts = tcga_raw_counts.set_index("gene_name")
else:
    tcga_raw_counts = tcga_raw_counts.copy()


sample_ids = tcga_raw_counts.columns.tolist()
patient_ids_from_counts = [sid[:12] for sid in sample_ids]

mapping_df = pd.DataFrame({
    "sample_id": sample_ids,
    "patient_id": patient_ids_from_counts
})

common_patients = set(mapping_df["patient_id"]).intersection(set(clinical_df["patient_id"]))
print(f"Number of common unique patients found: {len(common_patients)}")

clinical_cleaned = clinical_df[clinical_df["patient_id"].isin(common_patients)].copy()

clinical_cleaned = clinical_cleaned.merge(mapping_df, on="patient_id", how="inner")

clinical_cleaned = clinical_cleaned.sort_values("sample_id")

clinical_cleaned = clinical_cleaned.drop_duplicates(subset=["patient_id"], keep="first")
print(f"Clinical rows after strict deduplication: {clinical_cleaned.shape[0]}")

survival_sample_barcodes = clinical_cleaned["sample_id"].tolist()
counts_cleaned = tcga_raw_counts[survival_sample_barcodes].copy()

clinical_cleaned = clinical_cleaned.set_index("sample_id")
clinical_cleaned = clinical_cleaned.reindex(counts_cleaned.columns)

counts_cleaned.columns = [sid[:12] for sid in counts_cleaned.columns]

clinical_cleaned = clinical_cleaned.set_index("patient_id")

print("\n--- Final Matrix Verification ---")
print(f"Final Count Matrix Shape (Genes, Samples): {counts_cleaned.shape}")
print(f"Final Clinical Metadata Shape (Rows, Columns): {clinical_cleaned.shape}")


Number of common unique patients found: 725
Clinical rows after strict deduplication: 725

--- Final Matrix Verification ---
Final Count Matrix Shape (Genes, Samples): (19969, 725)
Final Clinical Metadata Shape (Rows, Columns): (725, 6)


### 8. Verify Dataset Integrity

In [8]:
assert counts_cleaned.shape[1] == clinical_cleaned.shape[0], "CRITICAL ERROR: Matrix dimensions mismatch!"

perfect_identity = all(counts_cleaned.columns == clinical_cleaned.index)
print(f"Are columns and indices perfectly identical strings? {perfect_identity}")

Are columns and indices perfectly identical strings? True


### 9. Export Clean Analysis Datasets

In [9]:
counts_cleaned.to_csv("../data/rna_seq/brca_counts_cleaned.csv")
clinical_cleaned.to_csv("../data/rna_seq/brca_clinical_cleaned.csv")